# Advanced E-Commerce User Behavior & Conversion Friction Analysis
### End-to-End Business & Data Analyst Project (BigQuery & BigFrames)

This notebook performs an **Advanced Exploratory Data Analysis (EDA)** on the Google Merchandise Store dataset (`data-to-insights.ecommerce.all_sessions`) containing **21.5 million rows**. 

We utilize Google Cloud's **BigFrames (BigQuery DataFrames)** library. BigFrames allows us to write standard pandas-like code that gets executed directly on Google Cloud's distributed SQL engine, bypassing the RAM limits of our local machine.

### Objectives:
1. **Authenticate and Initialize Connection** to BigQuery.
2. **Data Profiling & Quality Audit**: Assess missingness and structural columns.
3. **Friction Analysis (India Leakage)**: Deep dive into the conversion rates of US vs. India sessions.
4. **Shipping & Tax Fee Friction Analysis**: Audit average order value (AOV) and shipping/tax fees per country.
5. **Marketing Channel Efficacy**: Analyze which traffic sources yield the highest purchase conversion rate.
6. **Page Exit Share & Dropping Points**: Locate checkout hurdles and URL exits.
7. **Predictive Modeling**: Formulate a BigQuery ML (BQML) Purchase Propensity classification model.

In [ ]:
# 1. Environment Setup & Dependency Installation
# Install bigframes which contains pandas-like API for BigQuery compute
!pip install bigframes openpyxl pandas --quiet

## 2. Secure Authentication & BigFrames Connection Setup

We configure our Google Cloud Service Account key to authenticate and specify the project ID billing our compute. We then instantiate the `bigframes.pandas` context.

In [ ]:
import os
import pandas as pd
import bigframes.pandas as bpd

# Securely authenticate with the Google Cloud JSON key
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = 'valid-keep-465517-q8-32a662053b22.json'

# Initialize BigFrames options with our GCP Project ID
bpd.options.bigquery.project = 'valid-keep-465517-q8'
bpd.options.bigquery.location = 'US'

# Establish connection to the full dataset (21,493,109 rows)
# This is metadata-bound; data remains on Google Cloud storage.
df_full = bpd.read_gbq('data-to-insights.ecommerce.all_sessions')

print('Virtual Connection Established. Ready to process 21.5M rows on GCP!')

## 3. Data Profiling & Quality Audit

Let's look at the shape of the dataset and check the missingness percentages across columns to isolate dirty data.

In [ ]:
# Inspect the columns, data types, and non-null values
print('Columns and Data Types:')
print(df_full.dtypes)

# Calculate total volume of records
total_rows = len(df_full)
print(f'Total Records: {total_rows:,} rows')

In [ ]:
# Force GCP to compute missing percentages across the massive dataset
missing_counts = df_full.isnull().sum() / total_rows * 100
print('--- Missing Value Percentages per Column ---')
print(missing_counts.to_pandas().sort_values(ascending=False))

## 4. Friction Analysis: US vs. India conversion pipeline

We found that India has high session volumes but a very low conversion rate (0.09%). Let's construct a comparative dataframe showing where the leakage occurs between US and India visitors.

In [ ]:
# Query details specifically comparing United States vs India session milestones
pipeline_query = '''
WITH funnel_base AS (
  SELECT 
    country,
    COUNT(DISTINCT CONCAT(fullVisitorId, '_', visitId)) AS sessions,
    COUNT(DISTINCT CASE WHEN eCommerceAction_type = '2' THEN CONCAT(fullVisitorId, '_', visitId) END) AS views,
    COUNT(DISTINCT CASE WHEN eCommerceAction_type = '3' THEN CONCAT(fullVisitorId, '_', visitId) END) AS carts,
    COUNT(DISTINCT transactionId) AS orders
  FROM `data-to-insights.ecommerce.all_sessions`
  WHERE country IN ('United States', 'India')
  GROUP BY country
)
SELECT 
  country, 
  sessions AS `1. Session Start`,
  views AS `2. Product View`,
  carts AS `3. Add to Cart`,
  orders AS `4. Completed Order` 
FROM funnel_base;
'''

df_pipeline = bpd.read_gbq(pipeline_query)
df_pipeline_pd = df_pipeline.to_pandas()
print(df_pipeline_pd)

In [ ]:
# Transpose and calculate step-by-step dropoff rate
df_leak = df_pipeline_pd.set_index('country').T
df_leak['US_Dropoff_Pct'] = (1 - (df_leak['United States'] / df_leak['United States'].shift(1))) * 100
df_leak['India_Dropoff_Pct'] = (1 - (df_leak['India'] / df_leak['India'].shift(1))) * 100
print('--- Milestone Conversion & Dropoff Analysis ---')
print(df_leak)

**Key Finding**: India sees an enormous dropoff of **79.77%** from Session Start to Product View and a massive **98.98%** dropoff from Add to Cart to Completed Order, showing severe checkout friction.

## 5. Shipping & Tax Cost Friction Analysis (Geographic Fees)

Let's check the countries where the average shipping and tax cost (revenue minus product cost) is exceptionally high. High fees act as checkout blocks.

In [ ]:
fee_query = '''
WITH unique_orders AS (
  SELECT 
    country,
    transactionId,
    MAX(totalTransactionRevenue / 1000000) AS total_revenue_usd,
    MAX(COALESCE(transactionRevenue, totalTransactionRevenue) / 1000000) AS net_product_revenue_usd
  FROM `data-to-insights.ecommerce.all_sessions`
  WHERE transactionId IS NOT NULL
  GROUP BY country, transactionId
)
SELECT 
  country,
  COUNT(transactionId) AS total_orders,
  ROUND(AVG(total_revenue_usd), 2) AS avg_order_value_usd,
  ROUND(AVG(total_revenue_usd - net_product_revenue_usd), 2) AS avg_shipping_and_tax_usd,
  ROUND((SUM(total_revenue_usd - net_product_revenue_usd) / SUM(total_revenue_usd)) * 100, 2) AS fee_share_pct
FROM unique_orders
GROUP BY country
HAVING total_orders >= 5
ORDER BY fee_share_pct DESC;
'''

df_fees = bpd.read_gbq(fee_query)
print('--- Top Countries with Severe Shipping/Tax Fee Friction ---')
print(df_fees.to_pandas().head(10))

## 6. Marketing Channel Performance Audit

We identify which channels generate the most value and conversion success.

In [ ]:
channel_query = '''
WITH unique_sessions AS (
  SELECT  
    channelGrouping,
    CONCAT(fullVisitorId, '_', visitId) AS unique_session_id,
    MAX(CASE WHEN transactionId IS NOT NULL THEN 1 ELSE 0 END) AS is_confirmed_order,
    MAX(transactionId) AS unique_tx_id
  FROM `data-to-insights.ecommerce.all_sessions`
  GROUP BY channelGrouping, unique_session_id
)
SELECT  
  channelGrouping,
  COUNT(DISTINCT unique_session_id) AS total_visitors,
  COUNT(DISTINCT unique_tx_id) AS total_orders,
  ROUND((COUNT(DISTINCT unique_tx_id) / COUNT(DISTINCT unique_session_id)) * 100, 4) AS conversion_rate_pct,
  ROUND((COUNT(DISTINCT unique_tx_id) / SUM(COUNT(DISTINCT unique_tx_id)) OVER()) * 100, 2) AS order_share_pct
FROM unique_sessions
GROUP BY channelGrouping
ORDER BY total_orders DESC;
'''

df_channel = bpd.read_gbq(channel_query)
print('--- Channel Marketing Performance --- ')
print(df_channel.to_pandas())

## 7. Predictive Modeling: BigQuery ML Propensity Classifier

We build a Logistic Regression classification model inside BigQuery. We can command BigQuery to train this model and evaluate classification metrics.

In [ ]:
model_train_sql = '''
CREATE OR REPLACE MODEL `valid-keep-465517-q8.ecommerce.purchase_propensity_model`
OPTIONS(
  model_type='logistic_reg',
  input_label_cols=['label'],
  data_split_method='AUTO_SPLIT'
) AS
SELECT
  IF(transactionId IS NOT NULL, 1, 0) AS label,
  channelGrouping,
  country,
  IFNULL(pageviews, 0) AS pageviews,
  IFNULL(timeOnSite, 0) AS timeOnSite,
  IFNULL(sessionQualityDim, 0) AS session_quality_score
FROM `data-to-insights.ecommerce.all_sessions`
WHERE pageviews IS NOT NULL;
'''

print('Model creation SQL written. Run BQML query on BigQuery Console to train your model!')
print('Model name: valid-keep-465517-q8.ecommerce.purchase_propensity_model')

## 8. Summary of Analytical Findings & Next Steps

1. **Referrals are High Volume, but Low Profit Margins**: Referral channels drive the most volume, but the organic search and paid searches are more self-contained. 
2. **India Checkout Anomaly**: India represents a massive 25k traffic segment, but conversion is extremely depressed (0.09%). This points to local checkout friction (such as currency display, local card processing errors, or missing localized payment options like UPI).
3. **High Fee Friction in Specific Regions**: Countries like Indonesia (25.26% fee share) and Venezuela (59.46% fee share) suffer from severe shipping/tax fee ratios, highlighting regional logistics issues.